# 05 — Agents and Tools
Agents let the LLM decide which tool to call, observe the result, and loop until it has a final answer.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


## 1. Defining Custom Tools

In [ ]:
from langchain_core.tools import tool

@tool
def celsius_to_fahrenheit(celsius: float) -> str:
    """Converts a temperature from Celsius to Fahrenheit.
    Use when the user provides a Celsius temperature and wants Fahrenheit."""
    f = (celsius * 9 / 5) + 32
    return f"{celsius}°C = {f:.1f}°F"

@tool
def calculate(expression: str) -> str:
    """Evaluates a Python math expression like '2 ** 10' or '100 / 4'.
    Only supports basic arithmetic — no imports or function calls."""
    try:
        # Restrict builtins for safety
        result = eval(expression, {"__builtins__": {}}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

@tool
def word_count(text: str) -> str:
    """Counts the number of words in a given text string."""
    count = len(text.split())
    return f"The text contains {count} words."

# Test tools directly
print(celsius_to_fahrenheit.invoke({"celsius": 37}))
print(calculate.invoke({"expression": "2 ** 10"}))
print(word_count.invoke({"text": "LangChain makes building LLM apps easy"}))


## 2. Building a ReAct Agent

In [ ]:
from langchain.agents import AgentType, initialize_agent

tools = [celsius_to_fahrenheit, calculate, word_count]

# ZERO_SHOT_REACT_DESCRIPTION: model reads tool descriptions to decide which to use
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,          # shows Thought / Action / Observation loop
    max_iterations=5,      # safety limit on reasoning steps
    handle_parsing_errors=True
)

result = agent.run("What is 37 degrees Celsius in Fahrenheit? Also, what is 2 raised to the power 8?")
print("\nFinal Answer:", result)


## 3. Agent with Wikipedia Tool

In [ ]:
# pip install wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500))

research_agent = initialize_agent(
    tools=[wiki_tool, calculate],
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=4,
    handle_parsing_errors=True
)

result = research_agent.run("When was Python programming language created, and how many years ago was that?")
print("\nFinal Answer:", result)


## 4. Inspecting Tool Metadata

In [ ]:
for t in tools:
    print(f"Name       : {t.name}")
    print(f"Description: {t.description}")
    print(f"Args schema: {t.args}\n")


---
**Key takeaway:** The tool description is what the LLM reads to decide when to use it — write it like a docstring for a human collaborator, not just for the machine.